# EcoSignal Final Coverage Retraining v2

This Colab notebook retrains the lichen coverage model from scratch using the completed dataset.

Changes from the previous notebook:
- 20% untouched test set
- coverage-stratified splitting, including pure negatives
- 320×320 resolution
- lighter fine-tuning to reduce overfitting
- threshold selected to balance IoU and coverage MAE
- bootstrap confidence intervals


In [ ]:
!rm -rf /usr/local/lib/python3.13/dist-packages/~ensorflow*


In [ ]:
!pip -q install --force-reinstall -U tensorflow scikit-learn pillow pandas matplotlib

import json, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Please upload your zip files (e.g., `images.zip` and `masks.zip`) to your Colab environment. Once uploaded, run the following cells to extract them and configure the data paths.

In [ ]:
import zipfile
import os

# Define the paths for your zip files and extraction directories
# Replace 'images.zip' and 'masks.zip' with the actual names of your uploaded files
IMAGES_ZIP_PATH = 'Images.zip'
MASKS_ZIP_PATH = 'Masks.zip'

# Define the directories where the zip files will be extracted
# These will be temporary directories within the Colab environment
EXTRACTED_IMAGE_DIR = Path('/tmp/extracted_images')
EXTRACTED_MASK_DIR = Path('/tmp/extracted_masks')

# Create extraction directories if they don't exist
EXTRACTED_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
EXTRACTED_MASK_DIR.mkdir(parents=True, exist_ok=True)

# Extract images zip file
if os.path.exists(IMAGES_ZIP_PATH):
    with zipfile.ZipFile(IMAGES_ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACTED_IMAGE_DIR)
    print(f"Extracted {IMAGES_ZIP_PATH} to {EXTRACTED_IMAGE_DIR}")
else:
    print(f"Warning: {IMAGES_ZIP_PATH} not found. Please ensure you have uploaded it.")

# Extract masks zip file
if os.path.exists(MASKS_ZIP_PATH):
    with zipfile.ZipFile(MASKS_ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACTED_MASK_DIR)
    print(f"Extracted {MASKS_ZIP_PATH} to {EXTRACTED_MASK_DIR}")
else:
    print(f"Warning: {MASKS_ZIP_PATH} not found. Please ensure you have uploaded it.")


Now, let's update the `IMAGE_DIR` and `MASK_DIR` to use the extracted datasets. This will override the previous paths pointing to Google Drive.

In [ ]:
# Update IMAGE_DIR and MASK_DIR to use the extracted files
IMAGE_DIR = EXTRACTED_IMAGE_DIR
MASK_DIR = EXTRACTED_MASK_DIR

print(f"IMAGE_DIR set to: {IMAGE_DIR}")
print(f"MASK_DIR set to: {MASK_DIR}")


In [ ]:
PROJECT_DIR = Path("/content/drive/MyDrive/EcoSignalCoverage")
IMAGE_DIR = PROJECT_DIR / "images"
MASK_DIR = PROJECT_DIR / "masks"
OUTPUT_DIR = PROJECT_DIR / "training_output" / "final_retraining_v2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 320
BATCH_SIZE = 4
HEAD_EPOCHS = 18
FINE_TUNE_EPOCHS = 10
VAL_FRACTION = 0.15
TEST_FRACTION = 0.20
SEED = 42

EXTENSIONS = {".jpg",".jpeg",".png",".webp",".bmp",".tif",".tiff", ".avif"}

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


In [ ]:
IMAGE_DIR = EXTRACTED_IMAGE_DIR / "Images"
MASK_DIR = EXTRACTED_MASK_DIR / "Masks"

def files_by_stem(folder):
    return {
        p.stem: p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in EXTENSIONS
    }

images = files_by_stem(IMAGE_DIR)
mass = files_by_stem(MASK_DIR)
stems = sorted(set(images) & set(mass))

pairs = pd.DataFrame({
    "stem": stems,
    "image_path": [str(images[s]) for s in stems],
    "mask_path": [str(mass[s]) for s in stems],
})

print("Images:", len(images))
print("Masks:", len(mass))
print("Matched:", len(pairs))
print("Missing masks:", sorted(set(images)-set(mass))[:20])
print("Missing images:", sorted(set(mass)-set(images))[:20])

if len(pairs) < 40:
    raise ValueError("Too few matched image-mask pairs.")

In [ ]:
def mask_coverage(mask_path):
    mask = np.asarray(Image.open(mask_path).convert("L")) > 0
    return float(mask.mean() * 100)

pairs["coverage_percent"] = pairs["mask_path"].map(mask_coverage)

def coverage_bin(value):
    if value == 0: return "negative"
    if value < 10: return "very_low"
    if value < 30: return "low"
    if value < 60: return "medium"
    return "high"

pairs["coverage_bin"] = pairs["coverage_percent"].map(coverage_bin)

display(pairs["coverage_bin"].value_counts())
display(pairs["coverage_percent"].describe())

plt.figure(figsize=(8,4))
plt.hist(pairs["coverage_percent"], bins=20)
plt.xlabel("Ground-truth coverage (%)")
plt.ylabel("Images")
plt.show()


In [ ]:
def safe_labels(df):
    labels = df["coverage_bin"].copy()
    counts = labels.value_counts()

    # Identify categories with less than 3 samples to group into 'other'
    rare_initial = counts[counts < 3].index

    # Apply initial grouping
    processed_labels = labels.where(~labels.isin(rare_initial), "other")

    # Check the counts of the processed labels
    processed_counts = processed_labels.value_counts()

    # Identify if any class in the processed_labels still has less than 2 members
    problematic_classes = processed_counts[processed_counts < 2].index

    if len(problematic_classes) > 0:
        # If 'other' is a problematic class and has only 1 member, reassign it
        if "other" in problematic_classes and processed_counts["other"] == 1:
            # Find the largest non-'other' class to assign the singleton to
            largest_class = processed_counts[processed_counts.index != "other"].idxmax()

            # Find the index of the single 'other' sample
            other_index = processed_labels[processed_labels == "other"].index[0]

            # Reassign this single 'other' sample to the largest class
            processed_labels.loc[other_index] = largest_class

            print(f"Warning: 'other' class had only 1 member. Reassigned it to '{largest_class}' for stratification.")
        else:
            # If other problematic classes exist (e.g., another original class with 1 member after grouping),
            # or if 'other' has more than 1 but still less than 2 members, this case would need further review.
            # For simplicity, we assume 'other' is the only problematic class as per the error.
            pass # If other cases, current logic might not cover all edge cases.

    return processed_labels

labels = safe_labels(pairs)

train_df, temp_df = train_test_split(
    pairs,
    test_size=VAL_FRACTION + TEST_FRACTION,
    random_state=SEED,
    shuffle=True,
    stratify=labels,
)

temp_labels = safe_labels(temp_df)
relative_test = TEST_FRACTION / (VAL_FRACTION + TEST_FRACTION)

val_df, test_df = train_test_split(
    temp_df,
    test_size=relative_test,
    random_state=SEED,
    shuffle=True,
    stratify=temp_labels,
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

for name, df in [("Train",train_df),("Validation",val_df),("Test",test_df)]:
    print(name)
    display(df["coverage_bin"].value_counts())

with open(OUTPUT_DIR / "split_manifest.json","w") as f:
    json.dump({
        "train": train_df["stem"].tolist(),
        "validation": val_df["stem"].tolist(),
        "test": test_df["stem"].tolist(),
    }, f, indent=2)


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def read_image(path):
    data = tf.io.read_file(path)
    image = tf.io.decode_image(data, channels=3, expand_animations=False)
    image.set_shape([None,None,3])
    image = tf.image.convert_image_dtype(image, tf.float32)
    return tf.image.resize(image, [IMAGE_SIZE,IMAGE_SIZE], method="bilinear")

def read_mask(path):
    data = tf.io.read_file(path)
    mask = tf.io.decode_image(data, channels=1, expand_animations=False)
    mask.set_shape([None,None,1])
    mask = tf.cast(mask > 0, tf.float32)
    mask = tf.image.resize(mask, [IMAGE_SIZE,IMAGE_SIZE], method="nearest")
    return tf.cast(mask >= 0.5, tf.float32)

def load_pair(image_path, mask_path):
    return read_image(image_path), read_mask(mask_path)

def augment(image, mask):
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_left_right(image)
        mask = tf.image.flip_left_right(mask)
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_up_down(image)
        mask = tf.image.flip_up_down(mask)

    k = tf.random.uniform([],0,4,dtype=tf.int32)
    image = tf.image.rot90(image,k)
    mask = tf.image.rot90(mask,k)

    image = tf.image.random_brightness(image,0.08)
    image = tf.image.random_contrast(image,0.88,1.12)
    image = tf.image.random_saturation(image,0.88,1.12)
    image += tf.random.normal(tf.shape(image),stddev=0.01)
    image = tf.clip_by_value(image,0,1)
    return image,mask

def make_dataset(df, training=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (df["image_path"].values,df["mask_path"].values)
    )
    ds = ds.map(load_pair,num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(len(df),seed=SEED,reshuffle_each_iteration=True)
        ds = ds.map(augment,num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(train_df,True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)


In [ ]:
SMOOTH = 1e-6

def dice_coefficient(y_true,y_pred):
    yt = tf.reshape(tf.cast(y_true,tf.float32),[tf.shape(y_true)[0],-1])
    yp = tf.reshape(tf.cast(y_pred,tf.float32),[tf.shape(y_pred)[0],-1])
    inter = tf.reduce_sum(yt*yp,axis=1)
    denom = tf.reduce_sum(yt,axis=1)+tf.reduce_sum(yp,axis=1)
    return tf.reduce_mean((2*inter+SMOOTH)/(denom+SMOOTH))

def soft_iou(y_true,y_pred):
    yt = tf.reshape(tf.cast(y_true,tf.float32),[tf.shape(y_true)[0],-1])
    yp = tf.reshape(tf.cast(y_pred,tf.float32),[tf.shape(y_pred)[0],-1])
    inter = tf.reduce_sum(yt*yp,axis=1)
    union = tf.reduce_sum(yt,axis=1)+tf.reduce_sum(yp,axis=1)-inter
    return tf.reduce_mean((inter+SMOOTH)/(union+SMOOTH))

bce = keras.losses.BinaryCrossentropy()

def combined_loss(y_true,y_pred):
    return 0.5*bce(y_true,y_pred)+0.5*(1-dice_coefficient(y_true,y_pred))


In [ ]:
def decoder_block(x,skip,filters,name):
    x = layers.Conv2DTranspose(filters,3,strides=2,padding="same",
                               use_bias=False,name=name+"_up")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    if skip is not None:
        x = layers.Concatenate()([x,skip])
    for _ in range(2):
        x = layers.Conv2D(filters,3,padding="same",use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
    return x

def build_model():
    inputs = keras.Input((IMAGE_SIZE,IMAGE_SIZE,3))
    x0 = layers.Rescaling(2.0,offset=-1.0)(inputs)

    base = keras.applications.MobileNetV2(
        input_tensor=x0, include_top=False, weights="imagenet"
    )

    skip_names = [
        "block_1_expand_relu",
        "block_3_expand_relu",
        "block_6_expand_relu",
        "block_13_expand_relu",
    ]

    outputs = [base.get_layer(n).output for n in skip_names]
    outputs.append(base.get_layer("block_16_project").output)
    encoder = keras.Model(base.input,outputs,name="encoder")
    encoder.trainable = False

    s1,s2,s3,s4,x = encoder(inputs)
    x = decoder_block(x,s4,256,"d1")
    x = decoder_block(x,s3,128,"d2")
    x = decoder_block(x,s2,64,"d3")
    x = decoder_block(x,s1,32,"d4")
    x = decoder_block(x,None,16,"d5")
    output = layers.Conv2D(1,1,activation="sigmoid")(x)

    return keras.Model(inputs,output),encoder

model,encoder = build_model()

def compile_model(lr):
    model.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss=combined_loss,
        metrics=[
            dice_coefficient,
            soft_iou,
            keras.metrics.BinaryAccuracy(name="pixel_accuracy"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
        ],
    )

compile_model(1e-3)
model.summary()


In [ ]:
import io
from PIL import Image

# Helper function for read_image to be used with tf.py_function
def _read_image_py_logic(path_bytes_tensor):
    path_str = path_bytes_tensor.numpy().decode('utf-8')

    if any(path_str.lower().endswith(ext) for ext in [".tif", ".tiff", ".avif"]):
        img = Image.open(path_str).convert("RGB") # Ensure 3 channels
        img_array = np.array(img)
        image = tf.convert_to_tensor(img_array, dtype=tf.float32)
        image = image / 255.0 # Normalize pixel values to [0, 1]
    else:
        # For other formats, use Python's file open and then TF's decoder
        with open(path_str, 'rb') as f:
            image_bytes = f.read()
        image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
        image = tf.image.convert_image_dtype(image, tf.float32)

    # Set a known shape for the image tensor. The height/width can be dynamic.
    image.set_shape([None, None, 3])
    return image

# Redefine read_image to use tf.py_function
def read_image(path):
    image = tf.py_function(
        func=_read_image_py_logic,
        inp=[path],
        Tout=tf.float32
    )
    # The shape needs to be set explicitly after tf.py_function call for graph compatibility
    image.set_shape([None, None, 3])
    return tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE], method="bilinear")

# Helper function for read_mask to be used with tf.py_function
def _read_mask_py_logic(path_bytes_tensor):
    path_str = path_bytes_tensor.numpy().decode('utf-8')

    if any(path_str.lower().endswith(ext) for ext in [".tif", ".tiff", ".avif"]):
        mask_img = Image.open(path_str).convert("L") # Convert to grayscale
        mask_array = np.array(mask_img)
        mask = tf.convert_to_tensor(mask_array, dtype=tf.float32)
        mask = tf.expand_dims(mask, -1) # Add channel dimension
        mask = tf.cast(mask > 0, tf.float32) # Ensure binary mask
    else:
        # For other formats, use Python's file open and then TF's decoder
        with open(path_str, 'rb') as f:
            mask_bytes = f.read()
        mask = tf.io.decode_image(mask_bytes, channels=1, expand_animations=False)
        mask = tf.cast(mask > 0, tf.float32)

    # Set a known shape for the mask tensor
    mask.set_shape([None, None, 1])
    return mask

# Redefine read_mask to use tf.py_function
def read_mask(path):
    mask = tf.py_function(
        func=_read_mask_py_logic,
        inp=[path],
        Tout=tf.float32
    )
    # The shape needs to be set explicitly after tf.py_function call for graph compatibility
    mask.set_shape([None, None, 1])
    mask = tf.image.resize(mask, [IMAGE_SIZE, IMAGE_SIZE], method="nearest")
    return tf.cast(mask >= 0.5, tf.float32)

# Redefine load_pair as it depends on the now-overridden read_image and read_mask
def load_pair(image_path, mask_path):
    return read_image(image_path), read_mask(mask_path)

# Redefine make_dataset as it depends on the now-overridden load_pair
def make_dataset(df, training=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (df["image_path"].values,df["mask_path"].values)
    )
    ds = ds.map(load_pair,num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(len(df),seed=SEED,reshuffle_each_iteration=True)
        ds = ds.map(augment,num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

# Recreate the datasets with the fixed functions
train_ds = make_dataset(train_df,True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df) # Re-create test_ds for consistency, though not directly used here.

BEST_MODEL = OUTPUT_DIR / "best_model.keras"

callbacks = [
    keras.callbacks.ModelCheckpoint(
        str(BEST_MODEL),monitor="val_soft_iou",
        mode="max",save_best_only=True,verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_soft_iou",mode="max",
        patience=5,restore_best_weights=True,verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",factor=0.5,
        patience=2,min_lr=1e-6,verbose=1
    ),
    keras.callbacks.CSVLogger(str(OUTPUT_DIR/"head_history.csv")),
]

head_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=HEAD_EPOCHS,
    callbacks=callbacks,
)


In [ ]:
encoder.trainable = True
fine_tune_start = int(len(encoder.layers)*0.85)

for layer in encoder.layers[:fine_tune_start]:
    layer.trainable = False

for layer in encoder.layers:
    if isinstance(layer,layers.BatchNormalization):
        layer.trainable = False

compile_model(5e-6)

fine_callbacks = [
    keras.callbacks.ModelCheckpoint(
        str(BEST_MODEL),monitor="val_soft_iou",
        mode="max",save_best_only=True,verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_soft_iou",mode="max",
        patience=4,restore_best_weights=True,verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",factor=0.5,
        patience=2,min_lr=5e-8,verbose=1
    ),
    keras.callbacks.CSVLogger(str(OUTPUT_DIR/"fine_tune_history.csv")),
]

fine_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINE_TUNE_EPOCHS,
    callbacks=fine_callbacks,
)


In [ ]:
custom_objects = {
    "combined_loss":combined_loss,
    "dice_coefficient":dice_coefficient,
    "soft_iou":soft_iou,
}

model = keras.models.load_model(
    BEST_MODEL,
    custom_objects=custom_objects,
)


In [ ]:
def predict_dataframe(df):
    results = []
    for row in df.itertuples():
        image = read_image(row.image_path)
        mask = read_mask(row.mask_path)[...,0]
        prob = model.predict(image[None,...],verbose=0)[0,...,0]
        results.append({
            "stem":row.stem,
            "coverage_bin":row.coverage_bin,
            "image":image.numpy(),
            "mask":mask.numpy(),
            "probability":prob,
        })
    return results

val_predictions = predict_dataframe(val_df)
test_predictions = predict_dataframe(test_df)


In [ ]:
def image_metrics(true_mask,probability,threshold):
    true_mask = true_mask >= 0.5
    pred = probability >= threshold

    inter = np.logical_and(true_mask,pred).sum()
    union = np.logical_or(true_mask,pred).sum()
    tp = inter
    fp = np.logical_and(~true_mask,pred).sum()
    fn = np.logical_and(true_mask,~pred).sum()

    if true_mask.sum()==0 and pred.sum()==0:
        iou,dice = 1.0,1.0
    else:
        iou = inter/(union+1e-8)
        dice = 2*inter/(true_mask.sum()+pred.sum()+1e-8)

    true_cov = true_mask.mean()*100
    pred_cov = pred.mean()*100

    return {
        "iou":float(iou),
        "dice":float(dice),
        "precision":float(tp/(tp+fp+1e-8)),
        "recall":float(tp/(tp+fn+1e-8)),
        "true_coverage":float(true_cov),
        "predicted_coverage":float(pred_cov),
        "coverage_error":float(pred_cov-true_cov),
        "coverage_abs_error":float(abs(pred_cov-true_cov)),
    }


In [ ]:
rows = []

for threshold in np.arange(0.20,0.86,0.05):
    values = [
        image_metrics(x["mask"],x["probability"],float(threshold))
        for x in val_predictions
    ]
    rows.append({
        "threshold":float(threshold),
        "mean_iou":np.mean([v["iou"] for v in values]),
        "mean_dice":np.mean([v["dice"] for v in values]),
        "coverage_mae":np.mean([v["coverage_abs_error"] for v in values]),
    })

threshold_df = pd.DataFrame(rows)
display(threshold_df)

best_iou = threshold_df["mean_iou"].max()
candidates = threshold_df[
    threshold_df["mean_iou"] >= best_iou - 0.02
]

best_row = candidates.sort_values(
    ["coverage_mae","mean_iou"],
    ascending=[True,False],
).iloc[0]

BEST_THRESHOLD = float(best_row["threshold"])
print("Selected threshold:",BEST_THRESHOLD)

threshold_df.to_csv(OUTPUT_DIR/"threshold_sweep.csv",index=False)


In [ ]:
test_rows = []

for item in test_predictions:
    test_rows.append({
        "stem":item["stem"],
        "coverage_bin":item["coverage_bin"],
        **image_metrics(
            item["mask"],
            item["probability"],
            BEST_THRESHOLD,
        ),
    })

test_results = pd.DataFrame(test_rows)
display(test_results)

summary = {
    "test_images":int(len(test_results)),
    "threshold":BEST_THRESHOLD,
    "mean_iou":float(test_results["iou"].mean()),
    "median_iou":float(test_results["iou"].median()),
    "mean_dice":float(test_results["dice"].mean()),
    "mean_precision":float(test_results["precision"].mean()),
    "mean_recall":float(test_results["recall"].mean()),
    "coverage_mae_percentage_points":
        float(test_results["coverage_abs_error"].mean()),
    "coverage_bias_percentage_points":
        float(test_results["coverage_error"].mean()),
}

print(json.dumps(summary,indent=2))

test_results.to_csv(OUTPUT_DIR/"test_metrics.csv",index=False)
with open(OUTPUT_DIR/"test_summary.json","w") as f:
    json.dump(summary,f,indent=2)


In [ ]:
def bootstrap_ci(values,samples=5000):
    rng = np.random.default_rng(SEED)
    values = np.asarray(values)
    estimates = [
        np.mean(rng.choice(values,size=len(values),replace=True))
        for _ in range(samples)
    ]
    return [float(x) for x in np.percentile(estimates,[2.5,97.5])]

cis = {
    "mean_iou_95_ci":bootstrap_ci(test_results["iou"]),
    "mean_dice_95_ci":bootstrap_ci(test_results["dice"]),
    "coverage_mae_95_ci":
        bootstrap_ci(test_results["coverage_abs_error"]),
}

print(json.dumps(cis,indent=2))

with open(OUTPUT_DIR/"confidence_intervals.json","w") as f:
    json.dump(cis,f,indent=2)


In [ ]:
category_results = (
    test_results.groupby("coverage_bin")
    .agg(
        images=("stem","count"),
        mean_iou=("iou","mean"),
        mean_dice=("dice","mean"),
        coverage_mae=("coverage_abs_error","mean"),
        coverage_bias=("coverage_error","mean"),
    )
    .reset_index()
)

display(category_results)
category_results.to_csv(
    OUTPUT_DIR/"test_metrics_by_category.csv",
    index=False,
)


In [ ]:
positive_results = test_results[
    test_results["true_coverage"] > 0
].copy()

negative_results = test_results[
    test_results["true_coverage"] == 0
].copy()

positive_mean_iou = positive_results["iou"].mean()
positive_median_iou = positive_results["iou"].median()

negative_mean_false_coverage = (
    negative_results["predicted_coverage"].mean()
    if len(negative_results) > 0
    else 0.0
)

negative_below_2_percent_rate = (
    (negative_results["predicted_coverage"] < 2.0).mean()
    if len(negative_results) > 0
    else None
)

separated_summary = {
    "positive_test_images": int(len(positive_results)),
    "positive_mean_iou": float(positive_mean_iou),
    "positive_median_iou": float(positive_median_iou),
    "negative_test_images": int(len(negative_results)),
    "negative_mean_false_coverage_percent": float(
        negative_mean_false_coverage
    ),
    "negative_below_2_percent_rate": (
        float(negative_below_2_percent_rate)
        if negative_below_2_percent_rate is not None
        else None
    ),
    "overall_coverage_mae": float(
        test_results["coverage_abs_error"].mean()
    ),
}

print(json.dumps(separated_summary, indent=2))

In [ ]:
def show_prediction(item):
    m = image_metrics(
        item["mask"],
        item["probability"],
        BEST_THRESHOLD,
    )
    pred = item["probability"] >= BEST_THRESHOLD

    plt.figure(figsize=(16,4))
    plt.subplot(1,4,1)
    plt.imshow(item["image"])
    plt.title(item["stem"])
    plt.axis("off")

    plt.subplot(1,4,2)
    plt.imshow(item["mask"],cmap="gray")
    plt.title(f"True: {m['true_coverage']:.1f}%")
    plt.axis("off")

    plt.subplot(1,4,3)
    plt.imshow(item["probability"],cmap="gray",vmin=0,vmax=1)
    plt.title("Probability")
    plt.axis("off")

    plt.subplot(1,4,4)
    plt.imshow(pred,cmap="gray")
    plt.title(
        f"Pred: {m['predicted_coverage']:.1f}%\n"
        f"IoU: {m['iou']:.3f}"
    )
    plt.axis("off")
    plt.show()

worst = test_results.sort_values(
    ["iou","coverage_abs_error"],
    ascending=[True,False],
)

display(worst.head(12))

for stem in worst.head(min(12,len(worst)))["stem"]:
    item = next(x for x in test_predictions if x["stem"]==stem)
    show_prediction(item)


In [ ]:
FINAL_MODEL = OUTPUT_DIR/"lichen_coverage_model_v2.keras"
model.save(FINAL_MODEL)

metadata = {
    "image_size":IMAGE_SIZE,
    "threshold":BEST_THRESHOLD,
    "architecture":"MobileNetV2 U-Net",
    "training_images":len(train_df),
    "validation_images":len(val_df),
    "test_images":len(test_df),
    "test_summary":summary,
    "confidence_intervals":cis,
}

with open(OUTPUT_DIR/"model_metadata.json","w") as f:
    json.dump(metadata,f,indent=2)

print("Saved:",FINAL_MODEL)


In [ ]:
from google.colab import files

# Path to your final trained model
final_model_path = OUTPUT_DIR / "lichen_coverage_model_v2.keras"

# Check if the file exists before initiating download
if final_model_path.exists():
    print(f"Downloading {final_model_path.name} to your computer...")
    files.download(str(final_model_path))
else:
    print(f"Error: Model file not found at {final_model_path}. Make sure the training and saving cells have completed successfully.")

Validation w Apex Imgs

In [ ]:

import os
import hashlib
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from PIL import Image
from google.colab import drive, files

drive.mount("/content/drive")

# CHANGE THIS to the newly trained model's actual path.
MODEL_PATH = (
    "/content/drive/MyDrive/EcoSignalCoverage/"
    "training_output/final_retraining_v2/lichen_coverage_model_v2.keras"
)

assert os.path.isfile(MODEL_PATH), f"Not found: {MODEL_PATH}"

model = tf.keras.models.load_model(MODEL_PATH, compile=False)

print("Model:", MODEL_PATH)
print("Size (MiB):", round(os.path.getsize(MODEL_PATH) / 1024**2, 2))
print("Input:", model.input_shape)
print("Output:", model.output_shape)
print("Parameters:", model.count_params())

with open(MODEL_PATH, "rb") as f:
    print("SHA256:", hashlib.sha256(f.read()).hexdigest())

assert not isinstance(model.input_shape, list)
assert not isinstance(model.output_shape, list)
assert model.input_shape[-1] == 3
assert model.output_shape[-1] == 1

H, W = model.input_shape[1:3]
assert H is not None and W is not None


In [ ]:

uploaded = files.upload()

image_paths = []

for filename, content in uploaded.items():
    if filename.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
        path = os.path.join("/content", os.path.basename(filename))
        with open(path, "wb") as f:
            f.write(content)
        image_paths.append(path)

print("Loaded", len(image_paths), "images")


In [ ]:

THRESHOLD = 0.55  # Change if your NEW model uses another threshold

def validate_image(path, threshold=THRESHOLD):
    original = Image.open(path).convert("RGB")
    original_rgb = np.asarray(original)

    # Match the V2 preprocessing pipeline
    image = tf.convert_to_tensor(original_rgb)
    image = tf.image.convert_image_dtype(image, tf.float32)
    image = tf.image.resize(image, [H, W], method="bilinear")

    x = image.numpy()

    raw = np.asarray(
        model.predict(x[None, ...], verbose=0)
    )

    prob = np.squeeze(raw)

    if prob.shape != (H, W):
        raise ValueError(f"Unexpected prediction shape: {prob.shape}")

    if not np.isfinite(prob).all():
        raise ValueError("NaN or infinite model output")

    if prob.min() < 0 or prob.max() > 1:
        raise ValueError("Output may contain logits instead of probabilities")

    mask = prob >= threshold

    overlay = x.copy()
    overlay[mask] = (
        0.50 * overlay[mask]
        + 0.50 * np.array([1.0, 0.0, 0.0])
    )

    uncertain = (prob >= 0.4) & (prob <= 0.6)
    confidence = np.maximum(prob, 1 - prob)

    return {
        "original": original_rgb,
        "rgb": x,
        "prob": prob,
        "mask": mask,
        "overlay": np.clip(overlay, 0, 1),
        "coverage": float(mask.mean() * 100),
        "confidence": float(confidence.mean() * 100),
        "uncertain": float(uncertain.mean() * 100),
        "mean_prob": float(prob.mean())
    }


qa_results = {}

for path in image_paths:
    result = validate_image(path)
    qa_results[os.path.basename(path)] = result

    fig, axes = plt.subplots(1, 4, figsize=(19, 5))

    axes[0].imshow(result["rgb"])
    axes[0].set_title("Original Photograph")

    heatmap = axes[1].imshow(
        result["prob"], cmap="viridis", vmin=0, vmax=1
    )
    axes[1].set_title("Lichen Probability")
    fig.colorbar(heatmap, ax=axes[1], fraction=0.046)

    axes[2].imshow(result["mask"], cmap="gray", vmin=0, vmax=1)
    axes[2].set_title("Predicted Coverage Mask")

    axes[3].imshow(result["overlay"])
    axes[3].set_title(
        f"Coverage Overlay: {result["coverage"]:.2f}%"
    )

    for ax in axes:
        ax.axis("off")

    plt.suptitle(os.path.basename(path), fontsize=14)
    plt.tight_layout()
    plt.show()

    print("\nImage:", os.path.basename(path))
    print(f"Predicted coverage: {result["coverage"]:.2f}%")
    print(f"Mean probability: {result["mean_prob"]:.4f}")
    print(f"Mean prediction confidence: {result["confidence"]:.2f}%")
    print(f"Uncertain pixels (0.4–0.6): {result["uncertain"]:.2f}%")
    print("-" * 65)


## CAC stopping benchmark

Treat the model as ready when the untouched test set has:
- mean IoU ≥ 0.75
- median IoU ≥ 0.80
- coverage MAE < 5 percentage points
- at least about 20 test images
- no repeated failure category


### `test_summary.json`

In [ ]:
with open(OUTPUT_DIR / "test_summary.json", "r") as f:
    print(f.read())

### `test_metrics.csv`

In [ ]:
with open(OUTPUT_DIR / "test_metrics.csv", "r") as f:
    print(f.read())

Exports

### `test_metrics_by_category.csv`

In [ ]:
with open(OUTPUT_DIR / "test_metrics_by_category.csv", "r") as f:
    print(f.read())

### `threshold_sweep.csv`

In [ ]:
with open(OUTPUT_DIR / "threshold_sweep.csv", "r") as f:
    print(f.read())

### `confidence_intervals.json`

In [ ]:
with open(OUTPUT_DIR / "confidence_intervals.json", "r") as f:
    print(f.read())

### `head_history.csv`

In [ ]:
with open(OUTPUT_DIR / "head_history.csv", "r") as f:
    print(f.read())

### `fine_tune_history.csv`

In [ ]:
with open(OUTPUT_DIR / "fine_tune_history.csv", "r") as f:
    print(f.read())

### Download Output Files

In [ ]:
from google.colab import files

# Download test_summary.json
files.download(OUTPUT_DIR / "test_summary.json")

In [ ]:
from google.colab import files

# Download test_metrics.csv
files.download(OUTPUT_DIR / "test_metrics.csv")

In [ ]:
from google.colab import files

# Download test_metrics_by_category.csv
files.download(OUTPUT_DIR / "test_metrics_by_category.csv")

In [ ]:
from google.colab import files

# Download threshold_sweep.csv
files.download(OUTPUT_DIR / "threshold_sweep.csv")

In [ ]:
from google.colab import files

# Download confidence_intervals.json
files.download(OUTPUT_DIR / "confidence_intervals.json")

In [ ]:
from google.colab import files

# Download head_history.csv
files.download(OUTPUT_DIR / "head_history.csv")

In [ ]:
from google.colab import files

# Download fine_tune_history.csv
files.download(OUTPUT_DIR / "fine_tune_history.csv")

In [ ]:
from google.colab import files

files.download(OUTPUT_DIR / "lichen_coverage_model.keras")

In [ ]:
from google.colab import files

files.download(OUTPUT_DIR / "best_model.keras")

In [ ]:
from google.colab import files

files.download(OUTPUT_DIR / "model_metadata.json")

In [ ]:
from google.colab import files

files.download(OUTPUT_DIR / "split_manifest.json")

In [ ]:
from google.colab import files

files.download(OUTPUT_DIR / "test_metrics.csv")

In [ ]:
from google.colab import files

files.download(OUTPUT_DIR / "test_summary.json")

In [ ]:
from google.colab import files

files.download(OUTPUT_DIR / "threshold_sweep.csv")

Test Apex Img

In [ ]:
from PIL import Image, ImageOps

image_path = "/content/IMG_0792 (1).jpeg"

img = Image.open(image_path)
print("Original size:", img.size)
print("EXIF orientation:", img.getexif().get(274))

img = ImageOps.exif_transpose(img).convert("RGB")

print("Corrected size:", img.size)
print("Image mode:", img.mode)

display(img.resize((512, 512)))

In [ ]:
from PIL import Image, ImageOps
import matplotlib.pyplot as plt

img = ImageOps.exif_transpose(
    Image.open("/content/IMG_0792 (1).jpeg")
).convert("RGB")

fig, axes = plt.subplots(1, 3, figsize=(15, 6))

axes[0].imshow(img)
axes[0].set_title("Original (3024 × 4032)")

axes[1].imshow(img.resize((256, 256)))
axes[1].set_title("Resized to 256 × 256")

# Center crop at original resolution
w, h = img.size
crop = img.crop((
    w//2 - 128,
    h//2 - 128,
    w//2 + 128,
    h//2 + 128
))
axes[2].imshow(crop)
axes[2].set_title("256 × 256 Original-Resolution Crop")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
print("Model input:", model.input_shape)
print("Model output:", model.output_shape)

In [ ]:
from PIL import Image, ImageOps
import matplotlib.pyplot as plt

img = ImageOps.exif_transpose(
    Image.open("/content/IMG_0792 (1).jpeg")
).convert("RGB")

full_resize = img.resize((320, 320))

w, h = img.size
crop = img.crop((
    w // 2 - 160,
    h // 2 - 160,
    w // 2 + 160,
    h // 2 + 160
))

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(full_resize)
axes[0].set_title("Entire image resized to 320×320")

axes[1].imshow(crop)
axes[1].set_title("Original-resolution 320×320 tile")

for ax in axes:
    ax.axis("off")

plt.show()

In [ ]:
from PIL import Image, ImageOps
import numpy as np
import tensorflow as tf

def preprocess_apex(path):
    img = ImageOps.exif_transpose(
        Image.open(path)
    ).convert("RGB")

    x = np.asarray(img)

    x = tf.image.convert_image_dtype(
        tf.convert_to_tensor(x),
        tf.float32
    )

    x = tf.image.resize(
        x, [320, 320],
        method="bilinear"
    )

    return x.numpy()[None, ...]

x = preprocess_apex("/content/IMG_0792 (1).jpeg")

prediction = model.predict(x, verbose=0)[0, :, :, 0]

print("Probability range:", prediction.min(), prediction.max())
print("Coverage at 0.3:", (prediction >= 0.3).mean() * 100)
print("Coverage at 0.5:", (prediction >= 0.5).mean() * 100)
print("Coverage at 0.55:", (prediction >= 0.55).mean() * 100)

In [ ]:
from PIL import Image, ImageOps
import numpy as np
import matplotlib.pyplot as plt

IMAGE_PATH = "/content/IMG_0792 (1).jpeg"

img = ImageOps.exif_transpose(
    Image.open(IMAGE_PATH)
).convert("RGB")

original = np.asarray(img)
H, W = original.shape[:2]

TILE = 320
STRIDE = 240

prob_sum = np.zeros((H, W), dtype=np.float32)
weight_sum = np.zeros((H, W), dtype=np.float32)

def positions(length, tile, stride):
    pos = list(range(0, length - tile + 1, stride))
    if pos[-1] != length - tile:
        pos.append(length - tile)
    return pos

for y in positions(H, TILE, STRIDE):
    for x in positions(W, TILE, STRIDE):
        patch = original[y:y+TILE, x:x+TILE]

        # Match training RGB normalization
        patch_input = (
            patch.astype(np.float32) / 255.0
        )[None, ...]

        pred = model.predict(
            patch_input, verbose=0
        )[0, :, :, 0]

        prob_sum[y:y+TILE, x:x+TILE] += pred
        weight_sum[y:y+TILE, x:x+TILE] += 1

prob_map = prob_sum / np.maximum(weight_sum, 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 7))

axes[0].imshow(original)
axes[0].set_title("Original Apex image")

axes[1].imshow(prob_map, cmap="inferno", vmin=0, vmax=1)
axes[1].set_title("Tiled probability map")

axes[2].imshow(original)
axes[2].imshow(prob_map >= 0.5, cmap="spring", alpha=0.45)
axes[2].set_title("Predicted lichen mask at 0.5")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

for threshold in [0.3, 0.4, 0.5]:
    coverage = 100 * np.mean(prob_map >= threshold)
    print(f"Coverage @ {threshold}: {coverage:.2f}%")

print("Maximum probability:", prob_map.max())
print("Mean probability:", prob_map.mean())